# Chit 2 baseline: an autoencoder anomaly trigger · **FPGA-L1**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a quantized autoencoder on background events only, use its reconstruction error as the anomaly score, measure the signal efficiency (TPR) at the dataset's standard false-positive rate of 10⁻⁵ (~400 Hz of the 40 MHz collision rate), and check that the hls4ml emulation of the same network is bit-exact. For scale: the state of the art is ~4-5% on A→4ℓ; anything above 1% is good.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~2 min: it installs Brevitas and hls4ml) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["brevitas", "hls4ml==1.3.0", "quantizers<2", "qonnx", "onnxoptimizer", "onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data
Training is **background only, without labels**. Val and test mix 1M background events with four new-physics signals (label 1-4).

In [ ]:
from capstone.data.adc2021 import load, efficiency_at_fpr, SIGNALS, FPR
N_TRAIN = 50_000 if FAST else 500_000            # a subset of the 2M training events keeps training to a few minutes
X_tr, _ = load("train", max_items=N_TRAIN)
X_va, y_va = load("val")
X_te, y_te = load("test")
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape)
for k, s in enumerate(SIGNALS, start=1):
    print(f"  test: {s:<6} {np.sum(y_te == k):>7,} events")
print(f"  test: background {np.sum(y_te == 0):,}")

**Input scaling.** pT runs to hundreds of GeV, η and φ to a few units. We divide each **quantity** (all pT's, all η's, all φ's) by one **power of two**
chosen from the training split: the smallest one above its 99.9th percentile over non-empty slots (most slots are empty, i.e. zero). Most inputs then land
in [-1, 1]. On an FPGA a division by a power of two is free (a bit shift).

In [ ]:
KIND = np.arange(57) % 3                         # 0 = pT, 1 = eta, 2 = phi (slot by slot)
def pow2_scale(v):
    v = np.abs(v[v != 0])
    return 2.0 ** np.ceil(np.log2(np.quantile(v, 0.999)))
SCALE = np.array([pow2_scale(X_tr[:, KIND == k]) for k in range(3)])[KIND]
print("scales (pT, eta, phi):", SCALE[:3])
sc = lambda X: (X / SCALE).astype(np.float32)
Xtr, Xva, Xte = sc(X_tr), sc(X_va), sc(X_te)

## 2. The model, costed before training
A dense autoencoder 57 → 32 → 8 → 32 → 57, every layer 8-bit (weights and inputs), built with `qat_mlp`: the only builder that can be synthesized for the FPGA chits.

In [ ]:
from common.compress import qat_mlp
SIZES = (57, 32, 8, 32, 57)
CFG = [(8, 8)] * 4                               # (weight bits, input bits) per layer
model = qat_mlp(CFG, SIZES)
p = profile(model, (57,))
print(f"params {p['params']:,} | MACs {p['macs']:,} | weights at 8 bits {p['weight_bytes']:,} B")

## 3. Train on background, score by reconstruction error
Each feature's squared error is weighted by 1 / (its variance on the training split). Without that, the large pT values dominate the loss and the
"anomaly score" degenerates into "total pT". The same weights are used in the score (constants: cheap logic after the network).

In [ ]:
W = 1 / (Xtr.var(0) + 1e-4); W = (W / W.mean()).astype(np.float32)
Wt = torch.as_tensor(W)
fit(model, Xtr, Xtr, epochs=2 if FAST else 10, bs=1024, lr=3e-3, loss_fn=lambda out, target: ((out - target) ** 2 * Wt).mean())
score = lambda m, X: ((predict(m, X, bs=8192) - X) ** 2 * W).mean(1)      # anomaly score = weighted mean squared reconstruction error
EFF = efficiency_at_fpr(score(model, Xte), y_te)
for s, e in EFF.items():
    print(f"  TPR at FPR {FPR:g}: {s:<6} {e:.4f}")

**A non-ML reference to beat.** A trigger designer's first idea: count the leptons (electrons + muons) and break ties by their total pT.
No training, almost no logic. Is your network better than this, on every signal?

In [ ]:
pt = X_te[:, 0::3]
lep = (pt[:, 1:9] > 0).sum(1) + 1e-3 * pt[:, 1:9].sum(1)
REF = efficiency_at_fpr(lep, y_te)
for s, e in REF.items():
    print(f"  lepton count, TPR at FPR {FPR:g}: {s:<6} {e:.4f}")

## 4. On the FPGA: bit-exact emulation and what hls4ml builds
`to_hls` converts the Brevitas model; `check_bit_exact` compares Brevitas with the hls4ml C++ emulation; `dense_summary` lists the multipliers per layer.
Resources (DSP, LUT) and latency in cycles come from **C-synthesis**, which the instructors run for you (`common/submit.py`). Note that the anomaly score
(the squared error, summed) is extra logic after the network: cost it too.

In [ ]:
from common.hls import to_hls, check_bit_exact, dense_summary
hm, _ = to_hls(model, output_dir="hls_prj_adc2021", in_features=57)
diff, _ = check_bit_exact(hm, model, Xte[:2000])
print(f"max |Brevitas - hls4ml| on 2,000 test events: {diff:.3g}  ({'bit-exact' if diff == 0 else 'NOT bit-exact'})")
DS = dense_summary(hm)
for r in DS:
    print(r)
MULTS = sum(r["mults_needed"] for r in DS)

In [ ]:
ROW = report("2 adc2021", (len(Xtr), len(Xva), len(Xte)), (57,),
             f"TPR@FPR{FPR:g} " + ", ".join(f"{s} {100 * e:.2f}%" for s, e in EFF.items()) + " | lepton-count ref " + ", ".join(f"{100 * e:.2f}%" for e in REF.values()), p,
             f"FPGA RF=1: {MULTS:,} multipliers (dense_summary); cycles need C-synthesis; bit-exact={diff == 0}",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.